# Milestone 3

Retrieval-Augmented Generation (RAG), FAISS vector search, Cross-Encoders, and adversarial retrieval.

## Setup Run

Builds the knowledge base (correct answers from train.csv) and the FAISS index used in all questions below.

In [1]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 66.1 MB/s eta 0:00:00:00:0100:01


In [2]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [4]:
DATA_DIR = "/kaggle/input/competitions/smart-mcq-solver-challenge"

train = pd.read_csv(f"{DATA_DIR}/train.csv")

In [5]:
print("Creating knowledge base...")
kb = []
for idx, row in train.iterrows():
    correct_letter = row['answer']
    kb.append(str(row[correct_letter]))

print("Loading embedding model and creating index...")
model = SentenceTransformer('all-MiniLM-L6-v2')
kb_embeddings = model.encode(kb, show_progress_bar=False)
index = faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(kb_embeddings)

print(f"Knowledge base successfully created — {len(kb)} documents, index size: {index.ntotal}")

Creating knowledge base...
Loading embedding model and creating index...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Knowledge base successfully created — 2000 documents, index size: 2000


## Zero-shot classifier setup (shared across Q1, Q2, Q6)

In [6]:
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

row_150      = train.iloc[150]
prompt_150   = str(row_150['prompt'])
labels_150   = [str(row_150['A']), str(row_150['B']), str(row_150['C']),
                str(row_150['D']), str(row_150['E'])]
ans_150      = str(row_150[row_150['answer']])

print(f"Prompt 150 : {prompt_150}")
print(f"Correct opt: {row_150['answer']} — '{ans_150}'")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Prompt 150 : Select the most accurate option: What is the butterfly effect, as defined by Lorenz in his book "The Essence of Chaos"? based on the given context.
Correct opt: C — 'The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."'


## Q1. Zero-shot probability for the correct option at row 150

In [7]:
result_q1 = zs(prompt_150, labels_150)

# Find the score assigned to the ground-truth correct option text
score_correct = None
for label, score in zip(result_q1['labels'], result_q1['scores']):
    if label == ans_150:
        score_correct = score
        break

q1_answer = round(score_correct, 3)
print(f"Q1 — zero-shot scores:")
for l, s in zip(result_q1['labels'], result_q1['scores']):
    marker = " ← correct" if l == ans_150 else ""
    print(f"  {round(s,4)}  {l[:60]}{marker}")
print(f"\nQ1 answer: {q1_answer}")

Q1 — zero-shot scores:
  0.3844  The butterfly effect is the phenomenon that a small change i ← correct
  0.3787  The butterfly effect is the phenomenon that a large change i
  0.0927  The butterfly effect is the phenomenon that a small change i
  0.0786  The butterfly effect is the phenomenon that a small change i
  0.0656  The butterfly effect is the phenomenon that a large change i

Q1 answer: 0.384


## Q2. FAISS retrieval — rank of the true document for row 150

Embed the prompt at row 150 and query the FAISS index for the top k=10 most similar documents. Check where the true correct document (kb[150]) appears in the ranking.

In [8]:
prompt_150_emb = model.encode([prompt_150])

k = 10
distances, indices = index.search(prompt_150_emb, k)
retrieved_indices = indices[0].tolist()

print(f"Top-10 retrieved KB indices: {retrieved_indices}")
print(f"True document is at KB index: 150")

# Rank is 1-based position in the retrieved list
if 150 in retrieved_indices:
    q2_rank = retrieved_indices.index(150) + 1
else:
    q2_rank = "not in top 10"

print(f"\nQ2 answer — rank of true document: {q2_rank}")

Top-10 retrieved KB indices: [663, 1701, 1269, 1532, 576, 847, 1693, 1906, 168, 150]
True document is at KB index: 150

Q2 answer — rank of true document: 10


## Concept: Two-Stage Pipeline — Bi-encoder + Cross-Encoder

**Why FAISS alone can miss:** FAISS uses a Bi-encoder — the prompt and each document are embedded *separately*, then compared by distance. It's fast (scales to millions of docs) but treats each side independently, so it can miss nuanced semantic connections.

**Cross-Encoder fix:** Instead of embedding separately, a Cross-Encoder receives `[prompt, document]` concatenated as a single input. The attention mechanism can directly compare words across both sides, producing a much more accurate relevance score. The tradeoff: you can only run it on a small shortlist (top-k from FAISS), not the full KB.

**Two-stage pipeline:**
1. **Retrieve** — FAISS bi-encoder → fast top-k candidates
2. **Rerank** — Cross-Encoder → accurate ranking of those k candidates

## Q3. Cross-encoder reranking of the FAISS top-10 — new rank of the true document

In [9]:
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

docs_10 = [kb[i] for i in retrieved_indices]  # top-10 from Q2
pairs   = [[prompt_150, doc] for doc in docs_10]
ce_scores = cross_encoder.predict(pairs)

# Sort by cross-encoder score descending
reranked_order = np.argsort(-ce_scores).tolist()
reranked_kb_indices = [retrieved_indices[i] for i in reranked_order]

print("Cross-encoder reranked order (KB indices):")
for rank, (kb_idx, score) in enumerate(zip(reranked_kb_indices,
                                            sorted(ce_scores, reverse=True)), 1):
    marker = " ← TRUE DOCUMENT" if kb_idx == 150 else ""
    print(f"  Rank {rank}: KB[{kb_idx}]  score={score:.4f}{marker}")

q3_rank = reranked_kb_indices.index(150) + 1 if 150 in reranked_kb_indices else "not found"
print(f"\nQ3 answer — cross-encoder rank of true document: {q3_rank}")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Cross-encoder reranked order (KB indices):
  Rank 1: KB[150]  score=4.7585 ← TRUE DOCUMENT
  Rank 2: KB[1693]  score=4.7526
  Rank 3: KB[847]  score=4.7526
  Rank 4: KB[1906]  score=4.7526
  Rank 5: KB[1269]  score=4.7375
  Rank 6: KB[1532]  score=4.7375
  Rank 7: KB[168]  score=4.7072
  Rank 8: KB[576]  score=4.6870
  Rank 9: KB[1701]  score=4.6602
  Rank 10: KB[663]  score=4.6602

Q3 answer — cross-encoder rank of true document: 1


## Q4. Token count for RAG string at row index 42 with top k=5 documents

In [10]:
row_42      = train.iloc[42]
prompt_42   = str(row_42['prompt'])
prompt_42_emb = model.encode([prompt_42])

_, indices_42 = index.search(prompt_42_emb, 5)
docs_5_42 = [kb[i] for i in indices_42[0]]

# Concatenate docs with single space, then build the RAG string
concatenated = " ".join(docs_5_42)
rag_string_42 = f"Context: {concatenated} Question: {prompt_42}"

print(f"RAG string (first 300 chars):\n{rag_string_42[:300]}...\n")

bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Tokenize WITHOUT truncation
encoded_42 = bert_tokenizer(rag_string_42, truncation=False)
q4_answer = len(encoded_42['input_ids'])
print(f"Q4 answer — total tokens: {q4_answer}")

RAG string (first 300 chars):
Context: Permutation-inversion groups are groups of symmetry operations that are energetically feasible permutations of identical nuclei or inversion with respect to the center of mass, or a combination of both. Permutation-inversion groups are groups of symmetry operations that are energetically fe...



config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Q4 answer — total tokens: 216


## Q5. RAG with the TRUE document — zero-shot probability for the correct option at row 150

In [11]:
true_doc_150 = kb[150]  # the actual correct document for row 150

rag_string_150 = f"Context: {true_doc_150} Question: {prompt_150}"
print(f"RAG string:\n{rag_string_150}\n")

result_q5 = zs(rag_string_150, labels_150)

score_correct_q5 = None
for label, score in zip(result_q5['labels'], result_q5['scores']):
    if label == ans_150:
        score_correct_q5 = score
        break

q5_answer = round(score_correct_q5, 3)
print("Q5 — RAG zero-shot scores:")
for l, s in zip(result_q5['labels'], result_q5['scores']):
    marker = " ← correct" if l == ans_150 else ""
    print(f"  {round(s,4)}  {l[:60]}{marker}")
print(f"\nQ5 answer: {q5_answer}")
print(f"Change from Q1 ({q1_answer}) → Q5 ({q5_answer}): {round(q5_answer - q1_answer, 3):+.3f}")

RAG string:
Context: The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos." Question: Select the most accurate option: What is the butterfly effect, as defined by Lorenz in his book "The Essence of Chaos"? based on the given context.

Q5 — RAG zero-shot scores:
  0.9894  The butterfly effect is the phenomenon that a small change i ← correct
  0.0045  The butterfly effect is the phenomenon that a large change i
  0.0028  The butterfly effect is the phenomenon that a small change i
  0.0017  The butterfly effect is the phenomenon that a small change i
  0.0016  The butterfly effect is the phenomenon that a large change i

Q5 answer: 0.989
Change from Q1 (0.384) → Q5 (0.989): +0.605


## Concept: Adversarial RAG — Garbage In, Garbage Out

RAG injects external context directly into the model's prompt. The model is trained to trust context it's given — so if the retrieval step returns a wrong or unrelated document, the model often follows that bad context and produces a confident wrong answer. This is why reranking and filtering retrieved chunks matters so much.

## Q6. Adversarial RAG — force KB index 999 as context for row 150

In [12]:
adversarial_doc = kb[999]  # completely unrelated document

adv_rag_string = f"Context: {adversarial_doc} Question: {prompt_150}"
print(f"Adversarial context (KB[999]):\n'{adversarial_doc}'\n")
print(f"RAG string:\n{adv_rag_string}\n")

result_q6 = zs(adv_rag_string, labels_150)

score_correct_q6 = None
for label, score in zip(result_q6['labels'], result_q6['scores']):
    if label == ans_150:
        score_correct_q6 = score
        break

q6_answer = round(score_correct_q6, 3)
print("Q6 — Adversarial RAG zero-shot scores:")
for l, s in zip(result_q6['labels'], result_q6['scores']):
    marker = " ← correct" if l == ans_150 else ""
    print(f"  {round(s,4)}  {l[:60]}{marker}")
print(f"\nQ6 answer (adversarial): {q6_answer}")
print(f"\nScore comparison:")
print(f"  No context (Q1):           {q1_answer}")
print(f"  True context (Q5):         {q5_answer}")
print(f"  Adversarial context (Q6):  {q6_answer}")

Adversarial context (KB[999]):
'A thought experiment in which a demon guards a microscopic trapdoor in a wall separating two parts of a container filled with the same gas at equal temperatures. The demon selectively allows faster-than-average molecules to pass from one side to the other, causing a reduce in temperature in one part and an boost in temperature in the other, contrary to the second law of thermodynamics.'

RAG string:
Context: A thought experiment in which a demon guards a microscopic trapdoor in a wall separating two parts of a container filled with the same gas at equal temperatures. The demon selectively allows faster-than-average molecules to pass from one side to the other, causing a reduce in temperature in one part and an boost in temperature in the other, contrary to the second law of thermodynamics. Question: Select the most accurate option: What is the butterfly effect, as defined by Lorenz in his book "The Essence of Chaos"? based on the given context.

Q6 — Adv

## Q7. Hit Rate for first 100 rows — top k=5 retrieval

A 'hit' = the row's correct option text is found inside any of the 5 retrieved documents (exact string match).

In [13]:
hits = 0
N = 100
OPTION_COLS = ['A', 'B', 'C', 'D', 'E']

for i in range(N):
    row = train.iloc[i]
    prompt_i = str(row['prompt'])
    correct_text = str(row[row['answer']])  # the correct option's text

    # Embed and retrieve top-5
    emb_i = model.encode([prompt_i])
    _, idx_i = index.search(emb_i, 5)
    docs_5 = [kb[j] for j in idx_i[0]]

    # Hit if exact correct text appears in any retrieved doc
    if any(correct_text in doc for doc in docs_5):
        hits += 1

hit_rate = round(100 * hits / N, 1)
print(f"Hits: {hits} / {N}")
print(f"Q7 answer — Hit Rate: {hit_rate}%")

Hits: 73 / 100
Q7 answer — Hit Rate: 73.0%


## Q8. Full RAG pipeline MAP@3 on first 20 rows

Full pipeline per row: FAISS retrieve (k=5) → Cross-Encoder rerank → pick best doc → build RAG string → zero-shot classify → rank options → MAP@3.

In [14]:
def map_at_3_single(true_answer, top3_letters):
    if true_answer in top3_letters:
        return 1 / (top3_letters.index(true_answer) + 1)
    return 0

OPTION_COLS = ['A', 'B', 'C', 'D', 'E']
map3_scores = []

for i in range(20):
    row = train.iloc[i]
    prompt_i = str(row['prompt'])
    true_ans = str(row['answer'])
    options  = {opt: str(row[opt]) for opt in OPTION_COLS}
    labels_i = [options[opt] for opt in OPTION_COLS]

    # Step 1 — Retrieve top k=5 from FAISS
    emb_i = model.encode([prompt_i])
    _, idx_i = index.search(emb_i, 5)
    retrieved_idx_i = idx_i[0].tolist()
    docs_5_i = [kb[j] for j in retrieved_idx_i]

    # Step 2 — Rerank with Cross-Encoder
    pairs_i   = [[prompt_i, doc] for doc in docs_5_i]
    ce_i      = cross_encoder.predict(pairs_i)
    best_doc  = docs_5_i[int(np.argmax(ce_i))]

    # Step 3 — Build RAG string
    rag_i = f"Context: {best_doc} Question: {prompt_i}"

    # Step 4 — Zero-shot classify with 5 options as labels
    result_i = zs(rag_i, labels_i)

    # Step 5 — Map scores back to option letters, rank, take top-3 letters
    label_to_letter = {options[opt]: opt for opt in OPTION_COLS}
    scored = [(label_to_letter[label], score)
              for label, score in zip(result_i['labels'], result_i['scores'])]
    scored.sort(key=lambda x: -x[1])
    top3_letters = [letter for letter, _ in scored[:3]]

    # Score MAP@3
    score_i = map_at_3_single(true_ans, top3_letters)
    map3_scores.append(score_i)

    print(f"Row {i:2d} | true={true_ans} | top3={top3_letters} | MAP@3={score_i:.3f}")

q8_answer = round(np.mean(map3_scores), 3)
print(f"\nQ8 answer — average MAP@3 across 20 rows: {q8_answer}")

Row  0 | true=B | top3=['B', 'D', 'A'] | MAP@3=1.000
Row  1 | true=A | top3=['A', 'E', 'C'] | MAP@3=1.000
Row  2 | true=C | top3=['C', 'D', 'B'] | MAP@3=1.000
Row  3 | true=B | top3=['B', 'D', 'A'] | MAP@3=1.000
Row  4 | true=A | top3=['A', 'B', 'C'] | MAP@3=1.000
Row  5 | true=C | top3=['B', 'C', 'A'] | MAP@3=0.500
Row  6 | true=E | top3=['E', 'B', 'D'] | MAP@3=1.000
Row  7 | true=A | top3=['A', 'B', 'C'] | MAP@3=1.000
Row  8 | true=A | top3=['A', 'C', 'D'] | MAP@3=1.000
Row  9 | true=A | top3=['A', 'B', 'C'] | MAP@3=1.000
Row 10 | true=C | top3=['C', 'A', 'D'] | MAP@3=1.000
Row 11 | true=B | top3=['B', 'C', 'A'] | MAP@3=1.000
Row 12 | true=D | top3=['D', 'A', 'B'] | MAP@3=1.000
Row 13 | true=E | top3=['E', 'D', 'B'] | MAP@3=1.000
Row 14 | true=E | top3=['E', 'A', 'D'] | MAP@3=1.000
Row 15 | true=E | top3=['E', 'B', 'C'] | MAP@3=1.000
Row 16 | true=C | top3=['C', 'B', 'E'] | MAP@3=1.000
Row 17 | true=C | top3=['C', 'B', 'E'] | MAP@3=1.000
Row 18 | true=B | top3=['B', 'D', 'A'] | MAP@3